In [1]:
from fileinput import filename
import os

import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Optional
from collections import deque

In [2]:
# ── Data Containers ───────────────────────────────────────────────────────────

@dataclass
class OBLevel:
    price: float
    volume: float


@dataclass
class OBSnapshot:
    timestamp: int
    bids: list[OBLevel]   # sorted best first
    asks: list[OBLevel]   # sorted best first
    mid: float


@dataclass
class Trade:
    timestamp: int
    price: float
    quantity: float
    side: str             # "buy" | "sell" | "unknown"

In [3]:
# ── Tunable Parameters ────────────────────────────────────────────────────────

FACTOR_WEIGHTS = {
    "obi":         0.30,
    "vwmid_mom":   0.20,
    "tfi":         0.25,
    "spread_z":   -0.05,
    "depth_asym":  0.10,
    "price_mom":   0.10,
}

WINDOWS = {
    "obi_depth":       3,   # kept for flexibility; F5 uses all available depth
    "mom_fast":       10,
    "mom_slow":       30,
    "tfi_window":     80,   # TRUE tick window
    "spread_window":  50,
}


In [4]:
class FactorEngine:
    """
    Stateful engine: call update(ob, trades) every tick.
    update() only refreshes individual factor values.
    """

    def __init__(self):
        self._wmid_history: deque = deque(maxlen=WINDOWS["mom_slow"] + 5)
        self._mid_history: deque = deque(maxlen=WINDOWS["mom_slow"] + 5)
        self._spread_history: deque = deque(maxlen=WINDOWS["spread_window"] + 5)
        self._obi_history    = deque(maxlen=50)

        # FIX #1: true tick-based TFI window
        # store one signed volume per tick
        self._tfi_tick_window: deque = deque(maxlen=WINDOWS["tfi_window"])

        self.factors: dict = {}

    # ── Public API ────────────────────────────────────────────────────────────

    def update(self, ob: OBSnapshot, trades: list[Trade]) -> dict[str, float]:
        """
        Process one tick and refresh factor states only.
        trades: all trades whose timestamp == ob.timestamp
        """
        wmid = self._weighted_mid(ob)
        self._wmid_history.append(wmid)
        self._mid_history.append(ob.mid)

        spread = self._spread(ob)
        self._spread_history.append(spread)

        # aggregate current tick trade flow into one signed volume
        self._add_trades_one_tick(trades)

        f1 = self._f_obi(ob)
        f2 = self._f_vwmid_mom()
        f3 = self._f_tfi()
        f4 = self._f_spread_z()
        f5 = self._f_depth_asym(ob)
        f6 = self._f_price_mom()

        self.factors = {
            "obi":        f1,
            "vwmid_mom":  f2,
            "tfi":        f3,
            "spread_z":   f4,
            "depth_asym": f5,
            "price_mom":  f6,
        }

        return self.factors

    def compute_direction_score(self) -> float:
        """
        Combine current factors into one clipped score.
        """
        score = sum(FACTOR_WEIGHTS[k] * v for k, v in self.factors.items())
        return float(np.clip(score, -1.0, 1.0))
    
    # ── Factor Definitions ────────────────────────────────────────────────────

    def _f_obi(self, ob: OBSnapshot) -> float:
        """
        F1 — Order Book Imbalance

        1) Sum bid and ask volume over top N levels
        2) Compute imbalance = (bid_vol - ask_vol) / (bid_vol + ask_vol)
        3) Optionally smooth with a short moving average
        """

        OBI_LEVELS = 2   # include top 2 levels
        OBI_SMOOTH = 5   # moving-average window; set 1 to disable smoothing

        if not ob.bids or not ob.asks:
            return 0.0

        bid_vol = sum(lvl.volume for lvl in ob.bids[:OBI_LEVELS])
        ask_vol = sum(lvl.volume for lvl in ob.asks[:OBI_LEVELS])
        total = bid_vol + ask_vol
        raw_obi = ((bid_vol - ask_vol) / total) if total > 0 else 0.0

        self._obi_history.append(raw_obi)

        if OBI_SMOOTH <= 1:
            return float(raw_obi)

        recent = list(self._obi_history)[-OBI_SMOOTH:]
        return float(sum(recent) / len(recent))
    
    def _f_vwmid_mom(self) -> float:
        """
        F2 — Volume-Weighted Mid Momentum
        Fast vs slow mean difference on weighted mid, normalised by recent std.
        Range: tanh-squashed to (-1, +1)
        """
        if len(self._wmid_history) < WINDOWS["mom_fast"] + 1:
            return 0.0

        arr = np.array(self._wmid_history, dtype=float)
        fast = arr[-WINDOWS["mom_fast"]:].mean()
        slow = arr[-WINDOWS["mom_slow"]:].mean() if len(arr) >= WINDOWS["mom_slow"] else arr.mean()
        std = arr[:-1].std() + 1e-9  # baseline volatility from past-only window

        return float(np.tanh((fast - slow) / std))

    def _f_tfi(self) -> float:
        """
        F3 — Trade Flow Imbalance
        TRUE recent-20-tick net signed volume / total signed volume.
        Range: [-1, +1]
        """
        if not self._tfi_tick_window:
            return 0.0

        net_vol = sum(self._tfi_tick_window)
        total_vol = sum(abs(v) for v in self._tfi_tick_window)

        if total_vol == 0:
            return 0.0

        return float(np.clip(net_vol / total_vol, -1.0, 1.0))

    def _f_spread_z(self) -> float:
        """F4 — Spread Z-Score
        Current spread vs recent history, tanh-squashed to (-1, +1).
        """
        if len(self._spread_history) < 5:
            return 0.0
        arr     = np.array(self._spread_history, dtype=float)
        history = arr[:-1]          # exclude current tick from baseline
        if len(history) < 2:
            return 0.0
        mean = history.mean()
        std  = history.std() + 1e-9
        z    = (arr[-1] - mean) / std    # current tick vs past-only baseline
        return float(np.tanh(z))

    def _f_depth_asym(self, ob: OBSnapshot) -> float:
        """
        F5 — Total Depth Asymmetry
        Use all available depth (up to 3 levels).

        Range: [-1, +1]
        """
        bid_vol = sum(lvl.volume for lvl in ob.bids)
        ask_vol = sum(lvl.volume for lvl in ob.asks)
        total = bid_vol + ask_vol

        if total == 0:
            return 0.0

        return float((bid_vol - ask_vol) / total)

    def _f_price_mom(self) -> float:
        """
        F6 — Mid-Price EMA Crossover
        Fast EMA vs slow EMA on plain mid.
        Range: tanh-squashed to (-1, +1)
        """
        if len(self._mid_history) < WINDOWS["mom_fast"] + 1:
            return 0.0

        arr = np.array(self._mid_history, dtype=float)
        fast = self._ema(arr, WINDOWS["mom_fast"])
        slow = self._ema(arr, min(WINDOWS["mom_slow"], len(arr)))
        std = arr[:-1].std() + 1e-9  # baseline volatility from past-only window

        return float(np.tanh((fast - slow) / std))
    

    # ── Helpers ───────────────────────────────────────────────────────────────

    def _weighted_mid(self, ob: OBSnapshot) -> float:
        """
        Micro-price using best-level volume.
        """
        if not ob.bids or not ob.asks:
            return ob.mid

        bv = ob.bids[0].volume
        av = ob.asks[0].volume
        bp = ob.bids[0].price
        ap = ob.asks[0].price
        total = bv + av

        if total == 0:
            return (bp + ap) / 2.0

        return (ap * bv + bp * av) / total

    def _spread(self, ob: OBSnapshot) -> float:
        if ob.bids and ob.asks:
            return ob.asks[0].price - ob.bids[0].price
        return 0.0

    def _ema(self, arr: np.ndarray, span: int) -> float:
        alpha = 2.0 / (span + 1)
        result = arr[0]
        for v in arr[1:]:
            result = alpha * v + (1 - alpha) * result
        return float(result)

    def _add_trades_one_tick(self, trades: list[Trade]):
        """
        Aggregate all trades in this tick into one signed volume,
        then append to a TRUE tick-count rolling window.
        """
        signed_vol = 0.0

        for t in trades:
            if t.side == "buy":
                signed_vol += t.quantity
            elif t.side == "sell":
                signed_vol -= t.quantity

        self._tfi_tick_window.append(signed_vol)




In [5]:
# ── Data Loaders ──────────────────────────────────────────────────────────────

def load_order_book(path: str, product: str = "TOMATOES") -> pd.DataFrame:
    df = pd.read_csv(path, sep=";")
    df = df[df["product"] == product].copy()

    # Robust sort: some files may or may not have a 'day' column
    sort_cols = ["timestamp"]
    if "day" in df.columns:
        sort_cols = ["day", "timestamp"]

    df = df.sort_values(sort_cols).reset_index(drop=True)
    return df


def load_trades(path: str, product: str = "TOMATOES") -> pd.DataFrame:
    df = pd.read_csv(path, sep=";")
    df = df[df["symbol"] == product].copy()

    # Robust sort: some files may or may not have a 'day' column
    sort_cols = ["timestamp"]
    if "day" in df.columns:
        sort_cols = ["day", "timestamp"]

    df = df.sort_values(sort_cols).reset_index(drop=True)
    return df


def parse_ob_row(row) -> OBSnapshot:
    bids, asks = [], []

    for i in range(1, 4):
        bp = row.get(f"bid_price_{i}")
        bv = row.get(f"bid_volume_{i}")
        if pd.notna(bp) and pd.notna(bv) and float(bv) > 0:
            bids.append(OBLevel(price=float(bp), volume=float(bv)))

        ap = row.get(f"ask_price_{i}")
        av = row.get(f"ask_volume_{i}")
        if pd.notna(ap) and pd.notna(av) and float(av) > 0:
            asks.append(OBLevel(price=float(ap), volume=float(av)))

    return OBSnapshot(
        timestamp=int(row["timestamp"]),
        bids=bids,
        asks=asks,
        mid=float(row["mid_price"]),
    )


def infer_trade_side(price: float, ob: OBSnapshot) -> str:
    """
    Infer aggressor side from current order book snapshot.

    Rules:
      - if trade price >= best ask  -> buy aggressive
      - if trade price <= best bid  -> sell aggressive
      - else compare to mid:
          price > mid -> buy
          price < mid -> sell
          price == mid -> unknown
    """
    if not ob.bids or not ob.asks:
        return "unknown"

    best_bid = ob.bids[0].price
    best_ask = ob.asks[0].price
    mid = ob.mid

    if price >= best_ask:
        return "buy"
    elif price <= best_bid:
        return "sell"
    elif price >= mid:
        return "buy"
    else:
        return "sell"


def parse_trades_at(
    trades_df: pd.DataFrame,
    timestamp: int,
    ob: OBSnapshot,
    day: Optional[int] = None
) -> list[Trade]:
    """
    Return all trades at a given timestamp with side inferred from price vs book.
    """
    mask = trades_df["timestamp"] == timestamp
    if day is not None and "day" in trades_df.columns:
        mask = mask & (trades_df["day"] == day)
    rows = trades_df[mask]
    result = []

    for _, r in rows.iterrows():
        price = float(r["price"])
        qty = float(r["quantity"])
        side = infer_trade_side(price, ob)

        result.append(Trade(
            timestamp=int(r["timestamp"]),
            price=price,
            quantity=qty,
            side=side,
        ))

    return result

In [6]:
# ── Runner ────────────────────────────────────────────────────────────────────

def run(ob_path: str, trades_path: str, product: str = "TOMATOES") -> pd.DataFrame:
    """
    Run the model over historical data. Returns a DataFrame with one row per
    tick, containing all factor values and the final direction_score.

    Important:
    factor columns are shifted by 1 tick to avoid lookahead / alignment leakage.
    If a day column exists, shifting is done within each day independently.
    """
    ob_df = load_order_book(ob_path, product)
    trades_df = load_trades(trades_path, product)

    ob_has_day = "day" in ob_df.columns
    tr_has_day = "day" in trades_df.columns
    ob_multi_day = ob_has_day and ob_df["day"].nunique() > 1
    tr_multi_day = tr_has_day and trades_df["day"].nunique() > 1

    # Fail fast instead of silently matching trades across days by timestamp.
    if ob_multi_day and not tr_has_day:
        raise ValueError(
            "Order book has multiple days but trades file has no 'day' column. "
            "Add day to trades to avoid cross-day leakage."
        )
    if tr_multi_day and not ob_has_day:
        raise ValueError(
            "Trades file has multiple days but order book has no 'day' column. "
            "Provide day-aligned inputs to avoid cross-day leakage."
        )

    engine = FactorEngine()
    records = []

    for _, row in ob_df.iterrows():
        ts = int(row["timestamp"])
        ob = parse_ob_row(row)
        day = int(row["day"]) if "day" in row.index else None
        trades = parse_trades_at(trades_df, ts, ob, day=day)

        engine.update(ob, trades)
        score = engine.compute_direction_score()

        record = {
            "timestamp": ts,
            "mid_price": ob.mid,
            "F1_obi": engine.factors.get("obi", 0.0),
            "F2_vwmid_mom": engine.factors.get("vwmid_mom", 0.0),
            "F3_tfi": engine.factors.get("tfi", 0.0),
            "F4_spread_z": engine.factors.get("spread_z", 0.0),
            "F5_depth_asym": engine.factors.get("depth_asym", 0.0),
            "F6_price_mom": engine.factors.get("price_mom", 0.0),
            "direction_score": score,
        }

        if "day" in row.index:
            record["day"] = row["day"]

        records.append(record)

    out = pd.DataFrame(records)

    # sort again explicitly for safety
    sort_cols = ["timestamp"]
    if "day" in out.columns:
        sort_cols = ["day", "timestamp"]
    out = out.sort_values(sort_cols).reset_index(drop=True)

    # factor columns to be used for prediction / skew
    factor_cols = [
        "F1_obi",
        "F2_vwmid_mom",
        "F3_tfi",
        "F4_spread_z",
        "F5_depth_asym",
        "F6_price_mom",
        "direction_score",
    ]

    # shift by 1 tick to remove lookahead, without leaking across day boundaries
    if "day" in out.columns:
        out[factor_cols] = out.groupby("day", sort=False)[factor_cols].shift(1)
    else:
        out[factor_cols] = out[factor_cols].shift(1)

    # prefer day first if present
    cols = list(out.columns)
    if "day" in cols:
        ordered = [
            "day", "timestamp", "mid_price",
            "F1_obi", "F2_vwmid_mom", "F3_tfi",
            "F4_spread_z", "F5_depth_asym", "F6_price_mom",
            "direction_score",
        ]
        out = out[ordered]

    return out

In [7]:
# ── Generate Factor CSV ─────────────────────────────────────

def generate_and_save_factors(
    ob_path: str,
    trades_path: str,
    product: str = "TOMATOES",
    out_dir: str = "factors",
    preview_rows: int = 5,
) -> str:
    """
    Notebook-friendly wrapper:
    - runs factor generation
    - saves the factors CSV
    - prints save path and a small preview
    """
    results = run(ob_path, trades_path, product=product)

    os.makedirs(out_dir, exist_ok=True)
    filename = os.path.basename(ob_path)
    out_path = os.path.join(out_dir, filename.replace(".csv", "_factors.csv"))

    results.to_csv(out_path, index=False)

    print("Saved to:", out_path)
    print(results.head(preview_rows).to_string(index=False))
    return out_path


def generate_default_tutorial_factors() -> list[str]:
    """
    Convenience helper for the two tutorial tomato datasets.
    """
    jobs = [
        (
            "TUTORIAL_ROUND_1/prices_round_0_day_-2_tomatoes.csv",
            "TUTORIAL_ROUND_1/trades_round_0_day_-2_tomatoes.csv",
        ),
        (
            "TUTORIAL_ROUND_1/prices_round_0_day_-1_tomatoes.csv",
            "TUTORIAL_ROUND_1/trades_round_0_day_-1_tomatoes.csv",
        ),
    ]

    out_paths = []
    for ob_path, trades_path in jobs:
        out_paths.append(generate_and_save_factors(ob_path, trades_path))
    return out_paths


In [8]:
generate_default_tutorial_factors()

Saved to: factors/prices_round_0_day_-2_tomatoes_factors.csv
 day  timestamp  mid_price    F1_obi  F2_vwmid_mom  F3_tfi  F4_spread_z  F5_depth_asym  F6_price_mom  direction_score
  -2          0     5000.0       NaN           NaN     NaN          NaN            NaN           NaN              NaN
  -2        100     5002.5  0.000000           0.0     0.0          0.0       0.000000           0.0         0.000000
  -2        200     5001.0 -0.157143           0.0     0.0          0.0       0.098039           0.0        -0.037339
  -2        300     5001.5 -0.104762           0.0     0.0          0.0       0.000000           0.0        -0.031429
  -2        400     5001.5 -0.078571           0.0     0.0          0.0       0.000000           0.0        -0.023571
Saved to: factors/prices_round_0_day_-1_tomatoes_factors.csv
 day  timestamp  mid_price  F1_obi  F2_vwmid_mom  F3_tfi  F4_spread_z  F5_depth_asym  F6_price_mom  direction_score
  -1          0     5006.0     NaN           NaN     N

['factors/prices_round_0_day_-2_tomatoes_factors.csv',
 'factors/prices_round_0_day_-1_tomatoes_factors.csv']

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("factors/prices_round_0_day_-2_tomatoes_factors.csv")

# Horizon = 1: factor at t-1 predicts move from t-1 to t
df["move_h1"] = df["mid_price"] - df["mid_price"].shift(1)

# Horizon = 2: factor at t-1 predicts move from t-1 to t+1
df["move_h2"] = df["mid_price"].shift(-1) - df["mid_price"].shift(1)

# Horizon = 3: factor at t-1 predicts move from t-1 to t+2
df["move_h3"] = df["mid_price"].shift(-2) - df["mid_price"].shift(1)

ic_1 = df["F1_obi"].corr(df["move_h1"])
ic_2 = df["F2_vwmid_mom"].corr(df["move_h1"])
ic_3 = df["F3_tfi"].corr(df["move_h1"])
ic_4 = df["F4_spread_z"].corr(df["move_h1"])
ic_5 = df["F5_depth_asym"].corr(df["move_h1"])
ic_6 = df["F6_price_mom"].corr(df["move_h1"])

print("IC1:", ic_1, "IC2:", ic_2, "IC3:", ic_3, "IC4:", ic_4, "IC5:", ic_5, "IC6:", ic_6)


IC1: 0.2676196518447397 IC2: -0.03864618986712947 IC3: -0.0024935675182678 IC4: -0.03922623227833423 IC5: -0.5922084239750466 IC6: -0.10478526349933762


In [10]:
# IC for each factor (horizon = 2)

ic_1 = df["F1_obi"].corr(df["move_h2"])
ic_2 = df["F2_vwmid_mom"].corr(df["move_h2"])
ic_3 = df["F3_tfi"].corr(df["move_h2"])
ic_4 = df["F4_spread_z"].corr(df["move_h2"])
ic_5 = df["F5_depth_asym"].corr(df["move_h2"])
ic_6 = df["F6_price_mom"].corr(df["move_h2"])

print("IC1:", ic_1, "IC2:", ic_2, "IC3:", ic_3, "IC4:", ic_4, "IC5:", ic_5, "IC6:", ic_6)

IC1: 0.2498036203515212 IC2: -0.039351305741976164 IC3: -0.0045984051462766235 IC4: -0.03467565318664807 IC5: -0.5537361271714301 IC6: -0.10425590957356198


In [11]:
# IC for each factor (horizon = 3)
ic_1 = df["F1_obi"].corr(df["move_h3"])
ic_2 = df["F2_vwmid_mom"].corr(df["move_h3"])
ic_3 = df["F3_tfi"].corr(df["move_h3"])
ic_4 = df["F4_spread_z"].corr(df["move_h3"])
ic_5 = df["F5_depth_asym"].corr(df["move_h3"])
ic_6 = df["F6_price_mom"].corr(df["move_h3"])

print("IC1:", ic_1, "IC2:", ic_2, "IC3:", ic_3, "IC4:", ic_4, "IC5:", ic_5, "IC6:", ic_6)

IC1: 0.2349893313010114 IC2: -0.03936565850223343 IC3: -0.003975460398712024 IC4: -0.02575543256686282 IC5: -0.5228574566786551 IC6: -0.1038832461421896


In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("factors/prices_round_0_day_-1_tomatoes_factors.csv")

# Horizon = 1: factor at t-1 predicts move from t-1 to t
df["move_h1"] = df["mid_price"] - df["mid_price"].shift(1)

# Horizon = 2: factor at t-1 predicts move from t-1 to t+1
df["move_h2"] = df["mid_price"].shift(-1) - df["mid_price"].shift(1)

# Horizon = 3: factor at t-1 predicts move from t-1 to t+2
df["move_h3"] = df["mid_price"].shift(-2) - df["mid_price"].shift(1)

ic_1 = df["F1_obi"].corr(df["move_h1"])
ic_2 = df["F2_vwmid_mom"].corr(df["move_h1"])
ic_3 = df["F3_tfi"].corr(df["move_h1"])
ic_4 = df["F4_spread_z"].corr(df["move_h1"])
ic_5 = df["F5_depth_asym"].corr(df["move_h1"])
ic_6 = df["F6_price_mom"].corr(df["move_h1"])

print("IC1:", ic_1, "IC2:", ic_2, "IC3:", ic_3, "IC4:", ic_4, "IC5:", ic_5, "IC6:", ic_6)

IC1: 0.26824810790722414 IC2: -0.03345691142946735 IC3: 0.003908192958237082 IC4: -0.03416417689138808 IC5: -0.5802637177040663 IC6: -0.09693115122835196


In [13]:
# IC for each factor (horizon = 2)

ic_1 = df["F1_obi"].corr(df["move_h2"])
ic_2 = df["F2_vwmid_mom"].corr(df["move_h2"])
ic_3 = df["F3_tfi"].corr(df["move_h2"])
ic_4 = df["F4_spread_z"].corr(df["move_h2"])
ic_5 = df["F5_depth_asym"].corr(df["move_h2"])
ic_6 = df["F6_price_mom"].corr(df["move_h2"])

print("IC1:", ic_1, "IC2:", ic_2, "IC3:", ic_3, "IC4:", ic_4, "IC5:", ic_5, "IC6:", ic_6)

IC1: 0.2596363232574951 IC2: -0.03184434965048932 IC3: 0.004956579012195591 IC4: -0.025689088710536365 IC5: -0.5498528476012893 IC6: -0.09299220866924351


In [14]:
# IC for each factor (horizon = 3)

ic_1 = df["F1_obi"].corr(df["move_h3"])
ic_2 = df["F2_vwmid_mom"].corr(df["move_h3"])
ic_3 = df["F3_tfi"].corr(df["move_h3"])
ic_4 = df["F4_spread_z"].corr(df["move_h3"])
ic_5 = df["F5_depth_asym"].corr(df["move_h3"])
ic_6 = df["F6_price_mom"].corr(df["move_h3"])

print("IC1:", ic_1, "IC2:", ic_2, "IC3:", ic_3, "IC4:", ic_4, "IC5:", ic_5, "IC6:", ic_6)

IC1: 0.24564971220259452 IC2: -0.02945988762229665 IC3: 0.006425424074301663 IC4: -0.013494805478482681 IC5: -0.5208034033960807 IC6: -0.08889503230755184


Considering above, choose horizon = 1

In [15]:
df["move_h1"] = df["mid_price"] - df["mid_price"].shift(1)

ic = df["direction_score"].corr(df["move_h1"])

print("IC for combined score:", ic)

IC for combined score: -0.01946070516926814


In [16]:
# ===== Horizon = 2 calibrated weights =====
weights = {
    "obi":          0.013,    # 丢弃
    "vwmid_mom":   0.322,    # 保留，权重极小
    "tfi":         0.012,
    "spread_z":    0.188,    # 最强
    "depth_asym":   -0.070,    # 丢弃
    "price_mom":   -0.395,
}

df = pd.read_csv("factors/prices_round_0_day_-1_tomatoes_factors.csv")
df["direction_score"] = (
    weights["vwmid_mom"]  * df["F2_vwmid_mom"]
    + weights["tfi"]        * df["F3_tfi"]
    + weights["spread_z"]   * df["F4_spread_z"]
    + weights["depth_asym"] * df["F5_depth_asym"]
    + weights["price_mom"]  * df["F6_price_mom"]
)

df["direction_score"] = df["direction_score"].clip(-1.0, 1.0)

df = pd.read_csv("factors/prices_round_0_day_-2_tomatoes_factors.csv")
df["direction_score"] = (
    weights["vwmid_mom"]  * df["F2_vwmid_mom"]
    + weights["tfi"]        * df["F3_tfi"]
    + weights["spread_z"]   * df["F4_spread_z"]
    + weights["depth_asym"] * df["F5_depth_asym"]
    + weights["price_mom"]  * df["F6_price_mom"]
)

df["direction_score"] = df["direction_score"].clip(-1.0, 1.0)

In [17]:
df["next_move"] = df["mid_price"].shift(-2) - df["mid_price"]
ic = df["direction_score"].shift(-1).corr(df["next_move"])

print("IC for combined score:", ic)

IC for combined score: 0.07960465641268696


In [18]:
import pandas as pd

df = pd.read_csv("factors/prices_round_0_day_-2_tomatoes_factors.csv")
assert pd.isna(df.loc[0, "direction_score"])

df["move_h2"] = df["mid_price"].shift(-1) - df["mid_price"].shift(1)
print("k=0:", df["direction_score"].corr(df["move_h2"]))

for k in range(-8, 9):
    c = df["direction_score"].shift(k).corr(df["move_h2"])
    if abs(c) > 0.05:
        print("k=", k, "corr=", c)


k=0: -0.03170979200188429
k= -8 corr= 0.11143686559948694
k= -7 corr= 0.1006809623177802
k= -6 corr= 0.0737503738237771
k= -5 corr= 0.06178656554660075
k= -4 corr= 0.06184039005478322
